In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 定义生成器网络
class Generator:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.b2 = np.zeros((1, output_dim))

    def forward(self, z):
        self.z = z  # 保存输入数据
        self.z1 = np.dot(z, self.W1) + self.b1
        self.a1 = np.tanh(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = np.tanh(self.z2)
        return self.a2

    def backward(self, dz2):
        da2 = dz2 * (1 - self.a2 ** 2)
        dW2 = np.dot(self.a1.T, da2)
        db2 = np.sum(da2, axis=0, keepdims=True)
        da1 = np.dot(da2, self.W2.T) * (1 - self.a1 ** 2)
        dW1 = np.dot(self.z.T, da1)  # 使用保存的输入数据
        db1 = np.sum(da1, axis=0, keepdims=True)
        return dW1, db1, dW2, db2

# 定义判别器网络
class Discriminator:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.b2 = np.zeros((1, output_dim))

    def forward(self, x):
        self.x = x  # 保存输入数据
        self.z1 = np.dot(x, self.W1) + self.b1
        self.a1 = np.tanh(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = 1 / (1 + np.exp(-self.z2))  # Sigmoid
        return self.a2

    def backward(self, y_true, y_pred):
        da2 = y_pred - y_true
        dW2 = np.dot(self.a1.T, da2)
        db2 = np.sum(da2, axis=0, keepdims=True)
        da1 = np.dot(da2, self.W2.T) * (1 - self.a1 ** 2)
        dW1 = np.dot(self.x.T, da1)  # 使用保存的输入数据
        db1 = np.sum(da1, axis=0, keepdims=True)
        return dW1, db1, dW2, db2

# 训练GAN
def train_gan(generator, discriminator, num_epochs, batch_size, learning_rate):
    # 真实数据分布
    real_data = np.random.normal(0, 1, (1000, 1))

    for epoch in range(num_epochs):
        for _ in range(len(real_data) // batch_size):
            # 真实数据
            real_samples = real_data[np.random.randint(0, len(real_data), batch_size)]
            real_labels = np.ones((batch_size, 1))

            # 生成数据
            z = np.random.normal(0, 1, (batch_size, 1))
            fake_samples = generator.forward(z)
            fake_labels = np.zeros((batch_size, 1))

            # 合并数据
            all_samples = np.vstack((real_samples, fake_samples))
            all_labels = np.vstack((real_labels, fake_labels))

            # 训练判别器
            d_preds = discriminator.forward(all_samples)
            d_loss = -np.mean(all_labels * np.log(d_preds + 1e-7) + (1 - all_labels) * np.log(1 - d_preds + 1e-7))
            dW1, db1, dW2, db2 = discriminator.backward(all_labels, d_preds)
            discriminator.W1 -= learning_rate * dW1
            discriminator.b1 -= learning_rate * db1
            discriminator.W2 -= learning_rate * dW2
            discriminator.b2 -= learning_rate * db2

            # 训练生成器
            z = np.random.normal(0, 1, (batch_size, 1))
            fake_samples = generator.forward(z)
            d_preds_fake = discriminator.forward(fake_samples)
            g_loss = -np.mean(np.log(d_preds_fake + 1e-7))

            # 生成器的梯度计算
            dz2 = -1 / (d_preds_fake + 1e-7)
            dW1, db1, dW2, db2 = generator.backward(dz2)
            generator.W1 -= learning_rate * dW1
            generator.b1 -= learning_rate * db1
            generator.W2 -= learning_rate * dW2
            generator.b2 -= learning_rate * db2

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, D Loss: {d_loss}, G Loss: {g_loss}")

    # 生成一些样本并绘制
    z = np.random.normal(0, 1, (1000, 1))
    generated_samples = generator.forward(z)
    plt.hist(generated_samples, bins=30, alpha=0.5, label="Generated")
    plt.hist(real_data, bins=30, alpha=0.5, label="Real")
    plt.legend()
    plt.show()

# 初始化网络
generator = Generator(input_dim=1, hidden_dim=10, output_dim=1)
discriminator = Discriminator(input_dim=1, hidden_dim=10, output_dim=1)

# 训练GAN
train_gan(generator, discriminator, num_epochs=1000, batch_size=32, learning_rate=0.01)